# 03 – Basics expanded

### 1.

Libraries are collections of code that others have written. With import we can use them

In [ ]:
# import makes a library available; its functions are called with the name of the library and a dot
import math
print(math.sqrt(16))

In [ ]:
# with from ... import we take a single thing out of a library and can use it without the library name
# pathlib helps with files and folders: a Path describes where a file or folder is
from pathlib import Path

In [ ]:
# a library can get a shorter name with as; ElementTree reads XML files
import xml.etree.ElementTree as ET

### 2.

Our file is in Google Drive, in the folder `review-course/data/intern/XML-samples/`. To read it, we first connect the Drive

In [ ]:
# this library only exists in Colab; Google will ask for permission to access your Drive – allow it
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# the / joins the parts of a path, folder by folder
DATA_DIR = Path("/content/drive/MyDrive/review-course/data")
path = DATA_DIR / "intern" / "XML-samples" / "1001-SZ-01021985.xml"
print(path)

In [ ]:
# read_text() reads a file as one long string; the encoding tells python how the letters are stored (this file says so in its first line)
raw = path.read_text(encoding="iso-8859-1")
print(len(raw))
print(raw[:1500])

### 3.

The file is one issue of the Süddeutsche Zeitung, 1 February 1985, in XML. XML is text with tags: `<titel>` opens an element, `</titel>` closes it, and elements can contain other elements, like folders contain folders. Every `<artikel>` contains `<metadaten>` and `<inhalt>`, and `<inhalt>` contains the `<titel-liste>` and the `<text>`

In [ ]:
# ET.parse() reads the file and understands its structure; the root is the outermost element
tree = ET.parse(path)
root = tree.getroot()
print(root.tag)

In [ ]:
# findall() gives us a list of all elements with a certain tag
articles = root.findall("artikel")
print(len(articles))

In [ ]:
# it is an ordinary list, so positions work as usual; article number 212 is a theatre review
article = articles[212]
print(article.tag)

In [ ]:
# findtext() gives us the text inside an element; the path leads through the elements like through folders
print(article.findtext("metadaten/quelle/datum"))
print(article.findtext("inhalt/titel-liste/ressort"))
print(article.findtext("inhalt/titel-liste/titel"))

In [ ]:
# the text of an article consists of paragraphs (<absatz>)
paragraphs = article.findall("inhalt/text/absatz")
print(len(paragraphs))

In [ ]:
# some paragraphs contain further tags, like <b> for bold; itertext() collects all the text inside an element
# "".join() glues the pieces together into one string
for paragraph in paragraphs[:3]:
    print("".join(paragraph.itertext()))

### 4.

A dictionary holds pairs of keys and values: we look up a value by its key, like a word in a dictionary

In [ ]:
# a dictionary is written with curly brackets; each key is followed by a colon and its value
info = {"titel": article.findtext("inhalt/titel-liste/titel"), "ressort": article.findtext("inhalt/titel-liste/ressort")}
print(info)
print(info["ressort"])

In [ ]:
# a new pair is added by assigning a value to a new key
info["seite"] = article.findtext("metadaten/quelle/seite-start")
print(info)

In [ ]:
# keys() and values() give us all the keys or all the values; in checks the keys
print(info.keys())
print(info.values())
print("seite" in info)

In [ ]:
# a for-loop over a dictionary goes through its keys
for key in info:
    print(key, "=", info[key])

In [ ]:
# with a dictionary we can count: which sections (ressorts) does the newspaper have, and how many articles in each?
counts = {}
for article in articles:
    ressort = article.findtext("inhalt/titel-liste/ressort")
    if ressort in counts:
        counts[ressort] = counts[ressort] + 1
    else:
        counts[ressort] = 1

print(counts)

### 5.

A function is a piece of code with a name. We define it once and can then use it as often as we like

In [ ]:
# def defines a function; the variable in brackets is its argument, return gives back the result
def get_text(article):
    text = ""
    for paragraph in article.findall("inhalt/text/absatz"):
        text = text + "".join(paragraph.itertext()) + "\n"
    return text

In [ ]:
# the definition alone does nothing; the function only runs when it is called
print(get_text(articles[212])[:500])

In [ ]:
# a function can also return a dictionary; the date "01021985" is cut into pieces with slices
def get_info(article):
    datum = article.findtext("metadaten/quelle/datum")
    text = get_text(article)
    info = {
        "datum": datum[:2] + "." + datum[2:4] + "." + datum[4:],
        "seite": article.findtext("metadaten/quelle/seite-start"),
        "ressort": article.findtext("inhalt/titel-liste/ressort"),
        "rubrik": article.findtext("inhalt/titel-liste/rubrik"),
        "titel": article.findtext("inhalt/titel-liste/titel"),
        "woerter": len(text.split()),
        "text": text,
    }
    return info

In [ ]:
print(get_info(articles[100])["titel"])

In [ ]:
# now we collect the information of every article in a list of dictionaries
infos = []
for article in articles:
    infos.append(get_info(article))
print(len(infos))

In [ ]:
# the archive also contains every full page as an extra "article" (rubrik "Ganzseite"); we skip those and list the Feuilleton
for info in infos:
    if info["ressort"] == "Feuilleton" and info["rubrik"] != "Ganzseite":
        print(info["seite"], info["titel"], info["woerter"])

In [ ]:
# which of them talk about theatre?
for info in infos:
    if info["ressort"] == "Feuilleton" and info["rubrik"] != "Ganzseite" and "Theater" in info["text"]:
        print(info["titel"])
        print(info["text"][:400])
        print()

### 6.

Finally we write our results into a file, so that they stay in the Drive after Colab closes

In [ ]:
# we build a table as one string: "\t" (a tab) separates the columns, "\n" starts a new line
table = "datum\tseite\tressort\ttitel\twoerter\n"
for info in infos:
    if info["rubrik"] != "Ganzseite":
        table = table + info["datum"] + "\t" + info["seite"] + "\t" + info["ressort"] + "\t" + info["titel"] + "\t" + str(info["woerter"]) + "\n"

print(table[:1000])

In [ ]:
# write_text() writes a string into a file; careful: a file that already exists is overwritten
# a .tsv file (tab-separated values) can be opened with Google Sheets or Excel
out = DATA_DIR / "intern" / "SZ-01021985-overview.tsv"
out.write_text(table, encoding="utf-8")
print("written to", out)

In [ ]:
# to check, we read the file back in
print(out.read_text(encoding="utf-8")[:500])